# 02 – Lineare Algebra: Eigenwerte & PCA

**Modul:** 02 – Lineare Algebra  
**Thema:** Eigenwerte, Eigenvektoren und Principal Component Analysis (PCA)

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print('Bibliotheken geladen ✓')

## 1. Eigenwerte & Eigenvektoren

Ein Eigenvektor `v` einer Matrix `A` ist ein Vektor, der durch `A` nur skaliert (nicht rotiert) wird:

$$Av = \lambda v$$

Der Skalierungsfaktor `λ` ist der zugehörige Eigenwert.

In [ ]:
# Beispielmatrix
A = np.array([[3, 1],
              [1, 3]])

# Eigenwerte und Eigenvektoren berechnen
eigenvalues, eigenvectors = np.linalg.eig(A)

print(f'Matrix A:\n{A}')
print(f'\nEigenwerte: {eigenvalues}')
print(f'Eigenvektoren (Spalten):\n{eigenvectors}')

# Verifikation: Av = λv
v = eigenvectors[:, 0]
lam = eigenvalues[0]
print(f'\nVerifikation: Av = {A @ v}')
print(f'             λv = {lam * v}')

## 2. PCA von Grund auf implementieren

PCA nutzt Eigenvektoren der Kovarianzmatrix für Dimensionsreduktion.

In [ ]:
def pca_from_scratch(X, n_components):
    """
    PCA von Grund auf – ohne sklearn.
    
    Args:
        X: Datenmatrix (n_samples, n_features)
        n_components: Anzahl der Hauptkomponenten
    
    Returns:
        X_reduced: Projizierte Daten
        explained_variance_ratio: Erklärte Varianz pro Komponente
    """
    # Schritt 1: Zentrieren
    X_centered = X - X.mean(axis=0)
    
    # Schritt 2: Kovarianzmatrix
    cov_matrix = np.cov(X_centered.T)
    
    # Schritt 3: Eigenwertzerlegung
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    
    # Schritt 4: Absteigend sortieren
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # Schritt 5: Projektion
    components = eigenvectors[:, :n_components]
    X_reduced = X_centered @ components
    
    # Erklärte Varianz
    explained_variance_ratio = eigenvalues / eigenvalues.sum()
    
    return X_reduced, explained_variance_ratio[:n_components]

# Iris-Datensatz laden
iris = load_iris()
X, y = iris.data, iris.target

# PCA anwenden
X_2d, var_ratio = pca_from_scratch(X, n_components=2)

print(f'Original-Dimensionen: {X.shape}')
print(f'Reduzierte Dimensionen: {X_2d.shape}')
print(f'Erklärte Varianz: PC1={var_ratio[0]:.1%}, PC2={var_ratio[1]:.1%}')
print(f'Gesamt: {sum(var_ratio):.1%}')

In [ ]:
# Visualisierung
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#E74C3C', '#2ECC71', '#3498DB']
labels = iris.target_names

# PCA Scatter Plot
for i, (color, label) in enumerate(zip(colors, labels)):
    mask = y == i
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], 
                    c=color, label=label, alpha=0.7, s=60)
axes[0].set_xlabel(f'PC1 ({var_ratio[0]:.1%} Varianz)')
axes[0].set_ylabel(f'PC2 ({var_ratio[1]:.1%} Varianz)')
axes[0].set_title('PCA des Iris-Datensatzes')
axes[0].legend()

# Scree Plot
_, all_var = pca_from_scratch(X, n_components=4)
axes[1].bar(range(1, 5), all_var, color='#3498DB', alpha=0.8)
axes[1].plot(range(1, 5), np.cumsum(all_var), 'r-o', label='Kumulativ')
axes[1].set_xlabel('Hauptkomponente')
axes[1].set_ylabel('Erklärte Varianz')
axes[1].set_title('Scree Plot')
axes[1].legend()
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
plt.savefig('../resources/pca_iris.png', dpi=150)
plt.show()

## Übungsaufgaben

1. Berechne die Eigenwerte der Matrix `B = [[5,2],[2,2]]` von Hand und verifiziere mit NumPy.
2. Implementiere PCA mit SVD statt Eigenwertzerlegung und vergleiche die Ergebnisse.
3. Wende PCA auf den MNIST-Datensatz (sklearn) an und visualisiere die ersten 2 Komponenten.

→ Lösungen in `../solutions/02_pca_loesungen.ipynb`